# AIP Petrol Prices (Terminal Gate Prices)

In [1]:
import io
import re
import urllib.request
from pathlib import Path

import pandas as pd
import mgplot as mg

In [2]:
CHART_DIR = "./CHARTS/AIP/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False

In [3]:
# --- Fetch AIP Terminal Gate Price data (with caching) ---
CACHE_DIR = Path("./CACHE/AIP")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AIP_BASE = "https://www.aip.com.au/sites/default/files/download-files"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def _candidate_files():
    """Yield (filename, folder) for today back 5 days."""
    today = pd.Timestamp.today().normalize()
    for days_back in range(6):
        dt = today - pd.Timedelta(days=days_back)
        filename = f"AIP_TGP_Data_{dt.strftime('%-d-%b-%Y')}.xlsx"
        folder = dt.strftime("%Y-%m")
        yield filename, folder


def get_tgp_data() -> bytes:
    """Get AIP TGP Excel: check cache first, then download."""
    for filename, folder in _candidate_files():
        cache_path = CACHE_DIR / filename
        if cache_path.exists():
            print(f"Using cached: {filename}")
            return cache_path.read_bytes()

    for filename, folder in _candidate_files():
        url = f"{AIP_BASE}/{folder}/{filename}"
        req = urllib.request.Request(url, headers=HEADERS)
        try:
            data = urllib.request.urlopen(req, timeout=30).read()
            print(f"Downloaded: {filename}")
            (CACHE_DIR / filename).write_bytes(data)
            return data
        except urllib.error.HTTPError:
            continue

    raise FileNotFoundError("Could not find AIP TGP data file")


data = get_tgp_data()

petrol = pd.read_excel(
    io.BytesIO(data),
    sheet_name="Petrol TGP",
    header=0,
    index_col=0,
    parse_dates=True,
)
petrol.columns = petrol.columns.str.strip().str.replace("\n", " ")

diesel = pd.read_excel(
    io.BytesIO(data),
    sheet_name="Diesel TGP",
    header=0,
    index_col=0,
    parse_dates=True,
)
diesel.columns = diesel.columns.str.strip().str.replace("\n", " ")

print(f"Petrol: {petrol.index[0].date()} to {petrol.index[-1].date()}")
print(f"Diesel: {diesel.index[0].date()} to {diesel.index[-1].date()}")

Using cached: AIP_TGP_Data_20-Mar-2026.xlsx


Petrol: 2004-01-01 to 2026-03-20
Diesel: 2004-01-01 to 2026-03-20


In [4]:
# --- Filter to December 2025 onwards ---
START = "2025-12-01"
petrol_recent = petrol.loc[START:]
diesel_recent = diesel.loc[START:]

# Convert to PeriodIndex for mgplot
petrol_recent.index = pd.DatetimeIndex(petrol_recent.index).to_period("D")
diesel_recent.index = pd.DatetimeIndex(diesel_recent.index).to_period("D")

HORMUZ = {
    "x": pd.Period("2026-02-28", freq="D").ordinal,
    "color": "grey",
    "linestyle": "--",
    "linewidth": 1,
    "label": "Hormuz closure",
}
SOURCE = "Source: AIP"
data_to = petrol.index[-1].strftime("%-d-%b-%Y")
FOOTER = f"Australia. Daily wholesale terminal gate prices (inc. GST), national average, business days. Data to {data_to}."

In [5]:
# --- Chart: Petrol and Diesel national average TGPs ---
fuel_prices = pd.DataFrame({
    "Petrol (ULP)": petrol_recent["National Average"],
    "Diesel": diesel_recent["National Average"],
}).dropna()

mg.line_plot_finalise(
    fuel_prices,
    title="Fuel Terminal Gate Prices: Petrol and Diesel",
    ylabel="Cents per litre",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline=HORMUZ,
    lfooter=FOOTER,
    rfooter=SOURCE,
    show=SHOW,
)

print(f"Chart saved to {CHART_DIR}")

Chart saved to ./CHARTS/AIP/


In [6]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-03-20 19:56:14

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.11.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.21
pandas : 3.0.1
pathlib: 1.0.1
re     : 2.2.1

Watermark: 2.6.0



In [7]:
print("Finished")

Finished
